# 19. Generators, typed plans and coverage

Recipe → graph → plan → independent analysis.

In [1]:
import json
from pathlib import Path
import numpy as np
from QA_max_cut_lib import WeightedGraph
from QA_max_cut_lib.generators import GraphFactory, resolve_bench297
from QA_max_cut_lib.generators.plans import *
from QA_max_cut_lib.generators.structured import *
from QA_max_cut_lib.tools.partition_analysis import analyze_plan, hop_metrics
ROOT = Path.cwd()
if not (ROOT/'configs').exists(): ROOT = ROOT.parent
recipe = json.loads((ROOT/'configs/graphs/connected_path_fill_6.json').read_text())
graph = GraphFactory.from_recipe(recipe)
print(len(graph), len(graph.edges), graph.snapshot()['recipe']['resolved']['topology_hash'][:12])

6 9 eeb193c97d21


## Density G(n,m) differs from Bernoulli G(n,p)

In [2]:
g = GraphFactory.from_recipe({'schema_version':2,'type':'gnm_density_v1','seed':1,'parameters':{'n_vertices':4,'density':0.5}})
print('density edges:', len(g.edges), '(exact floor(0.5*6))')

density edges: 3 (exact floor(0.5*6))


## Benchmark297 decoder

In [3]:
for q in (17, 296): print(q, resolve_bench297(q))

17 {'benchmark_id': 17, 'family_index': 0, 'family': 'er_sparse', 'slot': 5, 'n_vertices': 200, 'repetition': 2, 'weight_index': 2, 'weights': 'signed_discrete', 'topology_seed': 50042, 'weight_seed': 50345, 'base_seed': 42, 'topology_group': 5, 'profile': 'benchmark297_legacy_v1'}
296 {'benchmark_id': 296, 'family_index': 10, 'family': 'complete', 'slot': 8, 'n_vertices': 300, 'repetition': 2, 'weight_index': 2, 'weights': 'signed_discrete', 'topology_seed': 980042, 'weight_seed': 980345, 'base_seed': 42, 'topology_group': 98, 'profile': 'benchmark297_legacy_v1'}


## Wishart integer witness

In [4]:
w, record = build_wishart_from_factor(np.array([[1.],[1.],[2.]]), [1,1,-1], labels=[80,-7,10])
print(w.edges); print(record['candidate_cut'], record['stored_graph_optimum_verified'])

((-7, 80, 1.0), (10, 80, 2.0), (-7, 10, 2.0))
4.0 False


## Hebbian counterexample

In [5]:
h, hr = build_hebbian_patterns([[1,1,1]]*5 + [[1,1,-1]])
print(h.edges); print(hr['candidates'][-1])

((0, 1, -5.0), (0, 2, -1.0), (1, 2, -1.0))
{'pattern_id': 5, 'coefficient': 6.0, 'candidate_cut': -2.0, 'candidate_energy': 2.0, 'optimality_status': 'not_certified'}


## Edge partition: the assigned domain is not induced

In [6]:
fixture = WeightedGraph([(-7,10,2),(10,30,-3),(-7,30,5)], nodes=[-7,10,30,80])
plan = build_overlap_edge_partition(fixture, max_size=3, seed=0)
analysis = analyze_plan(fixture, plan)
print(plan.plan_kind, analysis['edge_coverage'], analysis['pair_coverage'])
for part in plan.parts: print(part.nodes, part.edge_domain, len(part.edges))

edge_partition 1.0 0.5
(-7, 10, 30) explicit 3
(80,) explicit 0


## Hop metrics

In [7]:
path = WeightedGraph([(0,1,7),(1,2,-4)], nodes=[0,1,2])
print(hop_metrics(path))

{'mode': 'exact', 'n_pairs_universe': 3, 'n_pairs_requested': 3, 'n_pairs_reachable': 3, 'reachability_fraction': 1.0, 'reachable_mean_hop_distance': 1.3333333333333333, 'all_pairs_mean_hop_distance': 1.3333333333333333, 'global_efficiency': 0.8333333333333334, 'unreachable_pairs': 0, 'sampled': False, 'sample_seed': None, 'all_pairs_mean_reason': None}


## 48-case series

In [8]:
series = json.loads((ROOT/'results/generators_series/stage18_48_cases.json').read_text())
print(series['trials'], series['quantum_shots'])
from collections import Counter
print(Counter((row['builder'], row['completion']) for row in series['rows']))

48 0
Counter({('random', 'complete'): 12, ('bfs', 'complete'): 12, ('scan', 'complete'): 12, ('overlap', 'complete'): 12})
